# LOC Suspicious Entity Detection — Monthly Anomaly EDA

**Question**: Can we identify suspicious providers, hospital groups, or markets at **monthly** granularity using both cross-sectional and month-over-month anomaly detection?

**Source**: `tmp_1m.kn_loc_notif_05062026_od`  
**Population**: M&R FFS (`mnr_total_ffs_flag = 1`, `loc_flag = 1`, `ipa_pac_flag in ('IPA', 'MM')`)  
**Dimensions** (drill order): Market → Hospital Group → Provider TIN  
**Time range**: 202501–202604 (monthly)  
**Min cases per entity-month**: 15  

**Feature priority** (most → least important):  
1. persistent_adr_rate  
2. initial_adr_rate  
3. member_appeal_overturn_rate  
4. p2p_overturn_rate  
5. member_appeal_rate  
6. p2p_rate  
7. auth_per_k  

**Detection methods**:  
- Cross-sectional: MAD z-scores within each month (|z| > 2.5)  
- Month-over-month: Rolling z-score vs entity's own trailing 4-month baseline (need ≥6 months history)  
- Combined scoring: entities flagged by BOTH methods = highest priority

## 1. Setup & Configuration

In [ ]:
from itables import init_notebook_mode
init_notebook_mode(all_interactive = True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cmcrameri.cm as cmc
from pathlib import Path

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_style('whitegrid')

# Crameri scientific colour palette — colorblind-safe, perceptually uniform
ENTITY_COLORS = [cmc.batlowS(i) for i in range(10)]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color = ENTITY_COLORS)

OUTPUT_DIR = Path(r'C:\Users\knguy139\Documents\Projects\Data\Output')

SOURCE_TABLE = 'tmp_1m.kn_loc_notif_05062026_od'

# Dimension drill order: market -> hospital_group -> prov_tin
DIMENSIONS = ['fin_market', 'hospital_group', 'prov_tin']

MIN_CASES = 15

# Feature priority order (most -> least important)
PRIORITY_FEATURES = [
    'persistent_adr_rate',
    'initial_adr_rate',
    'member_appeal_overturn_rate',
    'p2p_overturn_rate',
    'member_appeal_rate',
    'p2p_rate',
    'auth_per_k',
]

# Flag direction: 'low' = suspicious if below peers, 'high' = suspicious if above
FLAG_DIRECTION = {
    'persistent_adr_rate': 'low',
    'initial_adr_rate': 'low',
    'member_appeal_overturn_rate': 'high',
    'p2p_overturn_rate': 'high',
    'member_appeal_rate': 'high',
    'p2p_rate': 'high',
    'auth_per_k': 'high',
}

# Cross-sectional threshold
MAD_Z_THRESHOLD = 2.5

# MoM parameters
MOM_ROLLING_WINDOW = 4   # trailing months
MOM_MIN_HISTORY = 6      # minimum months to compute rolling baseline
MOM_Z_THRESHOLD = 2.5

## 2. Snowflake Connection

In [ ]:
from sqlalchemy import create_engine
from snowflake.sqlalchemy import URL

engine = create_engine(URL(
    account = 'UHG-UHGDWAAS',
    user = 'KHANG.NGUYEN@UHC.COM',
    authenticator = 'externalbrowser',
    role = 'AZU_SDRP_VING_PRD_DEVELOPER_ROLE',
    warehouse = 'VING_PRD_MNR_HCE_DATAINFRA_WH',
    database = 'VING_PRD_TREND_DB',
    schema = 'TMP_1M',
))

## 3. Monthly Profiling — SQL Aggregation by (Dimension, Month)

Aggregation and rate computation pushed to Snowflake for speed.  
Each query returns pre-aggregated monthly profiles with `min_cases >= 15`.

In [ ]:
PROFILE_SQL_TEMPLATE = """
SELECT
    {dim},
    hce_admit_month,
    -- counts
    SUM(case_count) AS case_count,
    SUM(initial_adr_cnt) AS initial_adr_cnt,
    SUM(persistent_adr_cnt) AS persistent_adr_cnt,
    SUM(md_reviewed_cnt) AS md_reviewed_cnt,
    SUM(appeal_case_cnt) AS appeal_case_cnt,
    SUM(appeal_ovrtn_case_cnt) AS appeal_ovrtn_case_cnt,
    SUM(mcr_reconsideration_case_cnt) AS mcr_reconsideration_case_cnt,
    SUM(mcr_ovrtn_case_cnt) AS mcr_ovrtn_case_cnt,
    SUM(p2p_case_cnt) AS p2p_case_cnt,
    SUM(p2p_ovrtn_case_cnt) AS p2p_ovrtn_case_cnt,
    SUM(member_appeal_cnt) AS member_appeal_cnt,
    SUM(member_appeal_ovtn_cnt) AS member_appeal_ovtn_cnt,
    SUM(pre_auth_cases) AS pre_auth_cases,
    SUM(membership) AS membership,
    SUM(days) AS days,
    SUM(admits) AS admits,
    SUM(allowed) AS allowed,
    SUM(netpaid) AS netpaid,
    -- computed rates (priority order)
    SUM(persistent_adr_cnt) / NULLIF(SUM(case_count), 0) AS persistent_adr_rate,
    SUM(initial_adr_cnt) / NULLIF(SUM(case_count), 0) AS initial_adr_rate,
    SUM(member_appeal_ovtn_cnt) / NULLIF(SUM(initial_adr_cnt), 0) AS member_appeal_overturn_rate,
    SUM(p2p_ovrtn_case_cnt) / NULLIF(SUM(initial_adr_cnt), 0) AS p2p_overturn_rate,
    SUM(member_appeal_cnt) / NULLIF(SUM(initial_adr_cnt), 0) AS member_appeal_rate,
    SUM(p2p_case_cnt) / NULLIF(SUM(initial_adr_cnt), 0) AS p2p_rate,
    SUM(case_count) * 12000.0 / NULLIF(SUM(membership), 0) AS auth_per_k
FROM {source_table}
WHERE ipa_pac_flag IN ('IPA', 'MM')
    AND loc_flag = 1
    AND mnr_total_ffs_flag = 1
    AND hce_admit_month BETWEEN '202501' AND '202604'
GROUP BY {dim}, hce_admit_month
HAVING SUM(case_count) >= {min_cases}
ORDER BY {dim}, hce_admit_month
"""

monthly_profiles = {}

for dim in DIMENSIONS:
    sql = PROFILE_SQL_TEMPLATE.format(dim = dim, source_table = SOURCE_TABLE, min_cases = MIN_CASES)
    df = pd.read_sql(sql, engine)
    df.columns = df.columns.str.lower()
    monthly_profiles[dim] = df
    print(f'{dim}: {len(df):,} entity-months, {df[dim].nunique():,} entities, '
          f'{df["hce_admit_month"].nunique()} months')

# Summary
pd.DataFrame([
    {'dimension': dim, 'n_entity_months': len(df),
     'n_entities': df[dim].nunique(), 'n_months': df['hce_admit_month'].nunique()}
    for dim, df in monthly_profiles.items()
])

In [ ]:
# Available months (for later use in distribution profiling)
all_months = sorted(monthly_profiles['fin_market']['hce_admit_month'].unique())
print(f'Months available: {all_months[0]} — {all_months[-1]} ({len(all_months)} months)')

In [ ]:
engine.dispose()

In [ ]:
# Spot check: fin_market monthly profile
monthly_profiles['fin_market'][PRIORITY_FEATURES + ['fin_market', 'hce_admit_month', 'case_count']].describe().round(4)

## 4. Cross-Sectional Detection — MAD Z-Scores Within Each Month

For each (dimension, month), compute MAD-based z-scores for each feature.  
Flag directionally: `low` features (ADR rates) flag when z ≤ −2.5, `high` features (appeals/overturns/auth) flag when z ≥ 2.5.

**Drill order**: 4a. Market → 4b. Hospital Group → 4c. Provider TIN

In [ ]:
def mad_z_scores(series):
    """Compute MAD-based z-scores for a series."""
    median = series.median()
    mad = (series - median).abs().median()
    # 0.6745 is the scaling constant for consistency with normal distribution
    mad_scaled = mad * 1.4826
    if mad_scaled == 0:
        return pd.Series(0, index = series.index)
    return (series - median) / mad_scaled


def compute_cross_sectional_flags(profile_df, dim, features = PRIORITY_FEATURES, threshold = MAD_Z_THRESHOLD):
    """Compute MAD z-scores within each month for each feature. Flag directionally."""
    z_cols = [f'{f}_z' for f in features]
    
    results = []
    for month, grp in profile_df.groupby('hce_admit_month'):
        month_df = grp.copy()
        for feat in features:
            vals = month_df[feat].dropna()
            if len(vals) < 5:  # need enough peers for meaningful z-score
                month_df[f'{feat}_z'] = np.nan
            else:
                month_df[f'{feat}_z'] = mad_z_scores(month_df[feat])
        results.append(month_df)
    
    all_df = pd.concat(results, ignore_index = True)
    
    # Directional flagging: 'low' features flag on z <= -threshold, 'high' on z >= threshold
    for feat in features:
        direction = FLAG_DIRECTION[feat]
        if direction == 'low':
            all_df[f'{feat}_flag'] = all_df[f'{feat}_z'] <= -threshold
        else:
            all_df[f'{feat}_flag'] = all_df[f'{feat}_z'] >= threshold
    
    flag_cols = [f'{f}_flag' for f in features]
    all_df['cs_flagged'] = all_df[flag_cols].any(axis = 1)
    
    # Top feature: most extreme directional violation
    # For 'low' features use -z (most negative = most suspicious), for 'high' use +z
    directional_z = pd.DataFrame(index = all_df.index)
    for feat in features:
        direction = FLAG_DIRECTION[feat]
        if direction == 'low':
            directional_z[f'{feat}_z'] = -all_df[f'{feat}_z']  # flip so larger = more suspicious
        else:
            directional_z[f'{feat}_z'] = all_df[f'{feat}_z']
    
    all_df['max_abs_z'] = directional_z[z_cols].max(axis = 1)
    all_df['cs_top_feature'] = directional_z[z_cols].idxmax(axis = 1).str.replace('_z$', '', regex = True)
    
    # Drop intermediate flag columns
    all_df.drop(columns = flag_cols, inplace = True)
    
    return all_df


# Compute cross-sectional z-scores for all dimensions
cs_results = {}
for dim in DIMENSIONS:
    cs_results[dim] = compute_cross_sectional_flags(monthly_profiles[dim], dim)

# Summary of flagged entity-months
pd.DataFrame([
    {'dimension': dim, 'total_entity_months': len(df), 
     'flagged_entity_months': df['cs_flagged'].sum(),
     'pct_flagged': f"{df['cs_flagged'].mean()*100:.1f}%"}
    for dim, df in cs_results.items()
])

### 4a. Cross-Sectional — Market

In [ ]:
# Top flagged markets by frequency
cs_mkt = cs_results['fin_market']
cs_mkt_flagged = cs_mkt[cs_mkt['cs_flagged']].copy()

print(f'Flagged market-months: {len(cs_mkt_flagged)}')
print(f'Distinct flagged markets: {cs_mkt_flagged["fin_market"].nunique()}')

# Markets flagged most often
mkt_freq = (
    cs_mkt_flagged
    .groupby('fin_market')
    .agg(n_months_flagged = ('hce_admit_month', 'count'),
         avg_max_z = ('max_abs_z', 'mean'),
         top_features = ('cs_top_feature', lambda x: x.mode().iloc[0] if len(x) > 0 else ''))
    .sort_values('n_months_flagged', ascending = False)
    .head(15)
)
mkt_freq

In [ ]:
# Trend lines for top 5 flagged markets — priority features
top_mkts = mkt_freq.index[:5].tolist()

for feat in PRIORITY_FEATURES[:4]:  # top 4 features
    fig, ax = plt.subplots(figsize = (12, 4))
    for mkt in top_mkts:
        mkt_data = cs_mkt[cs_mkt['fin_market'] == mkt].sort_values('hce_admit_month')
        ax.plot(mkt_data['hce_admit_month'], mkt_data[feat], marker = 'o', label = mkt, markersize = 4)
    ax.set_title(f'{feat} — Top Flagged Markets (Cross-Sectional)')
    ax.set_xlabel('Month')
    ax.set_ylabel(feat)
    ax.legend(loc = 'upper left', fontsize = 8)
    plt.xticks(rotation = 45)
    plt.tight_layout()
    plt.show()

### 4b. Cross-Sectional — Hospital Group

In [ ]:
cs_hg = cs_results['hospital_group']
cs_hg_flagged = cs_hg[cs_hg['cs_flagged']].copy()

print(f'Flagged hospital_group-months: {len(cs_hg_flagged)}')
print(f'Distinct flagged hospital_groups: {cs_hg_flagged["hospital_group"].nunique()}')

hg_freq = (
    cs_hg_flagged
    .groupby('hospital_group')
    .agg(n_months_flagged = ('hce_admit_month', 'count'),
         avg_max_z = ('max_abs_z', 'mean'),
         avg_cases = ('case_count', 'mean'),
         top_features = ('cs_top_feature', lambda x: x.mode().iloc[0] if len(x) > 0 else ''))
    .sort_values('n_months_flagged', ascending = False)
    .head(15)
)
hg_freq

In [ ]:
# Trend lines for top 5 flagged hospital groups
top_hgs = hg_freq.index[:5].tolist()

for feat in PRIORITY_FEATURES[:4]:
    fig, ax = plt.subplots(figsize = (12, 4))
    for hg in top_hgs:
        hg_data = cs_hg[cs_hg['hospital_group'] == hg].sort_values('hce_admit_month')
        ax.plot(hg_data['hce_admit_month'], hg_data[feat], marker = 'o', label = hg[:30], markersize = 4)
    ax.set_title(f'{feat} — Top Flagged Hospital Groups (Cross-Sectional)')
    ax.set_xlabel('Month')
    ax.set_ylabel(feat)
    ax.legend(loc = 'upper left', fontsize = 8)
    plt.xticks(rotation = 45)
    plt.tight_layout()
    plt.show()

### 4c. Cross-Sectional — Provider TIN

In [ ]:
cs_tin = cs_results['prov_tin']
cs_tin_flagged = cs_tin[cs_tin['cs_flagged']].copy()

print(f'Flagged prov_tin-months: {len(cs_tin_flagged)}')
print(f'Distinct flagged prov_tins: {cs_tin_flagged["prov_tin"].nunique()}')

tin_freq = (
    cs_tin_flagged
    .groupby('prov_tin')
    .agg(n_months_flagged = ('hce_admit_month', 'count'),
         avg_max_z = ('max_abs_z', 'mean'),
         avg_cases = ('case_count', 'mean'),
         top_features = ('cs_top_feature', lambda x: x.mode().iloc[0] if len(x) > 0 else ''))
    .sort_values('n_months_flagged', ascending = False)
    .head(20)
)
tin_freq

In [ ]:
# Trend lines for top 5 flagged TINs
top_tins = tin_freq.index[:5].tolist()

for feat in PRIORITY_FEATURES[:4]:
    fig, ax = plt.subplots(figsize = (12, 4))
    for tin in top_tins:
        tin_data = cs_tin[cs_tin['prov_tin'] == tin].sort_values('hce_admit_month')
        ax.plot(tin_data['hce_admit_month'], tin_data[feat], marker = 'o', label = str(tin), markersize = 4)
    ax.set_title(f'{feat} — Top Flagged Provider TINs (Cross-Sectional)')
    ax.set_xlabel('Month')
    ax.set_ylabel(feat)
    ax.legend(loc = 'upper left', fontsize = 8)
    plt.xticks(rotation = 45)
    plt.tight_layout()
    plt.show()

## 5. Month-over-Month Detection — Rolling Z-Score vs Own Baseline

For each entity, compute a rolling mean/std over the trailing 4 months, then z-score the current month against that baseline.  
Entities need ≥6 months history to have enough data for meaningful baselines.

**Drill order**: 5a. Market → 5b. Hospital Group → 5c. Provider TIN

In [ ]:
def compute_mom_flags(profile_df, dim, features = PRIORITY_FEATURES,
                      window = MOM_ROLLING_WINDOW, min_history = MOM_MIN_HISTORY,
                      threshold = MOM_Z_THRESHOLD):
    """Compute MoM rolling z-scores for each entity. Flag directionally vs own history."""
    results = []
    
    # Only entities with enough months of history
    entity_months = profile_df.groupby(dim)['hce_admit_month'].nunique()
    eligible_entities = entity_months[entity_months >= min_history].index
    eligible_df = profile_df[profile_df[dim].isin(eligible_entities)].copy()
    
    for entity_id, entity_df in eligible_df.groupby(dim):
        entity_sorted = entity_df.sort_values('hce_admit_month').copy()
        
        for feat in features:
            roll_mean = entity_sorted[feat].rolling(window = window, min_periods = 3).mean().shift(1)
            roll_std = entity_sorted[feat].rolling(window = window, min_periods = 3).std().shift(1)
            # Avoid division by zero
            roll_std = roll_std.replace(0, np.nan)
            entity_sorted[f'{feat}_mom_z'] = (entity_sorted[feat] - roll_mean) / roll_std
        
        results.append(entity_sorted)
    
    if not results:
        return pd.DataFrame()
    
    all_df = pd.concat(results, ignore_index = True)
    
    # Directional flagging: 'low' features flag on mom_z <= -threshold, 'high' on mom_z >= threshold
    mom_z_cols = [f'{f}_mom_z' for f in features]
    
    for feat in features:
        direction = FLAG_DIRECTION[feat]
        if direction == 'low':
            all_df[f'{feat}_mom_flag'] = all_df[f'{feat}_mom_z'] <= -threshold
        else:
            all_df[f'{feat}_mom_flag'] = all_df[f'{feat}_mom_z'] >= threshold
    
    mom_flag_cols = [f'{f}_mom_flag' for f in features]
    all_df['mom_flagged'] = all_df[mom_flag_cols].any(axis = 1)
    
    # Top feature: most extreme directional violation
    directional_mom_z = pd.DataFrame(index = all_df.index)
    for feat in features:
        direction = FLAG_DIRECTION[feat]
        if direction == 'low':
            directional_mom_z[f'{feat}_mom_z'] = -all_df[f'{feat}_mom_z']
        else:
            directional_mom_z[f'{feat}_mom_z'] = all_df[f'{feat}_mom_z']
    
    all_df['max_abs_mom_z'] = directional_mom_z[mom_z_cols].max(axis = 1)
    all_df['mom_top_feature'] = directional_mom_z[mom_z_cols].idxmax(axis = 1).str.replace('_mom_z$', '', regex = True)
    
    # Drop intermediate flag columns
    all_df.drop(columns = mom_flag_cols, inplace = True)
    
    return all_df


# Compute MoM z-scores for all dimensions
mom_results = {}
for dim in DIMENSIONS:
    mom_results[dim] = compute_mom_flags(monthly_profiles[dim], dim)

# Summary
pd.DataFrame([
    {'dimension': dim, 'eligible_entity_months': len(df),
     'eligible_entities': df[dim].nunique() if len(df) > 0 else 0,
     'flagged_entity_months': df['mom_flagged'].sum() if len(df) > 0 else 0,
     'pct_flagged': f"{df['mom_flagged'].mean()*100:.1f}%" if len(df) > 0 else '0%'}
    for dim, df in mom_results.items()
])

### 5a. MoM — Market

In [ ]:
mom_mkt = mom_results['fin_market']
if len(mom_mkt) > 0:
    mom_mkt_flagged = mom_mkt[mom_mkt['mom_flagged']].copy()
    print(f'MoM flagged market-months: {len(mom_mkt_flagged)}')
    print(f'Distinct flagged markets: {mom_mkt_flagged["fin_market"].nunique()}')
    
    mom_mkt_freq = (
        mom_mkt_flagged
        .groupby('fin_market')
        .agg(n_months_flagged = ('hce_admit_month', 'count'),
             avg_mom_z = ('max_abs_mom_z', 'mean'),
             top_features = ('mom_top_feature', lambda x: x.mode().iloc[0] if len(x) > 0 else ''))
        .sort_values('n_months_flagged', ascending = False)
        .head(15)
    )
    display(mom_mkt_freq)
else:
    print('No eligible markets (need >= 6 months history)')

In [ ]:
# MoM trend for top flagged markets
if len(mom_mkt) > 0 and len(mom_mkt_flagged) > 0:
    top_mom_mkts = mom_mkt_freq.index[:5].tolist()
    
    for feat in PRIORITY_FEATURES[:4]:
        fig, ax = plt.subplots(figsize = (12, 4))
        for mkt in top_mom_mkts:
            mkt_data = mom_mkt[mom_mkt['fin_market'] == mkt].sort_values('hce_admit_month')
            ax.plot(mkt_data['hce_admit_month'], mkt_data[feat], marker = 'o', label = mkt, markersize = 4)
        ax.set_title(f'{feat} — Top MoM-Flagged Markets')
        ax.set_xlabel('Month')
        ax.set_ylabel(feat)
        ax.legend(loc = 'upper left', fontsize = 8)
        plt.xticks(rotation = 45)
        plt.tight_layout()
        plt.show()

### 5b. MoM — Hospital Group

In [ ]:
mom_hg = mom_results['hospital_group']
if len(mom_hg) > 0:
    mom_hg_flagged = mom_hg[mom_hg['mom_flagged']].copy()
    print(f'MoM flagged hospital_group-months: {len(mom_hg_flagged)}')
    print(f'Distinct flagged hospital_groups: {mom_hg_flagged["hospital_group"].nunique()}')
    
    mom_hg_freq = (
        mom_hg_flagged
        .groupby('hospital_group')
        .agg(n_months_flagged = ('hce_admit_month', 'count'),
             avg_mom_z = ('max_abs_mom_z', 'mean'),
             avg_cases = ('case_count', 'mean'),
             top_features = ('mom_top_feature', lambda x: x.mode().iloc[0] if len(x) > 0 else ''))
        .sort_values('n_months_flagged', ascending = False)
        .head(15)
    )
    display(mom_hg_freq)
else:
    print('No eligible hospital groups')

In [ ]:
# MoM trend for top flagged hospital groups
if len(mom_hg) > 0 and len(mom_hg_flagged) > 0:
    top_mom_hgs = mom_hg_freq.index[:5].tolist()
    
    for feat in PRIORITY_FEATURES[:4]:
        fig, ax = plt.subplots(figsize = (12, 4))
        for hg in top_mom_hgs:
            hg_data = mom_hg[mom_hg['hospital_group'] == hg].sort_values('hce_admit_month')
            ax.plot(hg_data['hce_admit_month'], hg_data[feat], marker = 'o', label = hg[:30], markersize = 4)
        ax.set_title(f'{feat} — Top MoM-Flagged Hospital Groups')
        ax.set_xlabel('Month')
        ax.set_ylabel(feat)
        ax.legend(loc = 'upper left', fontsize = 8)
        plt.xticks(rotation = 45)
        plt.tight_layout()
        plt.show()

### 5c. MoM — Provider TIN

In [ ]:
mom_tin = mom_results['prov_tin']
if len(mom_tin) > 0:
    mom_tin_flagged = mom_tin[mom_tin['mom_flagged']].copy()
    print(f'MoM flagged prov_tin-months: {len(mom_tin_flagged)}')
    print(f'Distinct flagged prov_tins: {mom_tin_flagged["prov_tin"].nunique()}')
    
    mom_tin_freq = (
        mom_tin_flagged
        .groupby('prov_tin')
        .agg(n_months_flagged = ('hce_admit_month', 'count'),
             avg_mom_z = ('max_abs_mom_z', 'mean'),
             avg_cases = ('case_count', 'mean'),
             top_features = ('mom_top_feature', lambda x: x.mode().iloc[0] if len(x) > 0 else ''))
        .sort_values('n_months_flagged', ascending = False)
        .head(20)
    )
    display(mom_tin_freq)
else:
    print('No eligible provider TINs')

In [ ]:
# MoM trend for top flagged TINs
if len(mom_tin) > 0 and len(mom_tin_flagged) > 0:
    top_mom_tins = mom_tin_freq.index[:5].tolist()
    
    for feat in PRIORITY_FEATURES[:4]:
        fig, ax = plt.subplots(figsize = (12, 4))
        for tin in top_mom_tins:
            tin_data = mom_tin[mom_tin['prov_tin'] == tin].sort_values('hce_admit_month')
            ax.plot(tin_data['hce_admit_month'], tin_data[feat], marker = 'o', label = str(tin), markersize = 4)
        ax.set_title(f'{feat} — Top MoM-Flagged Provider TINs')
        ax.set_xlabel('Month')
        ax.set_ylabel(feat)
        ax.legend(loc = 'upper left', fontsize = 8)
        plt.xticks(rotation = 45)
        plt.tight_layout()
        plt.show()

## 6. Combined Scoring — Entities Flagged by BOTH Methods

Entities flagged by both cross-sectional AND MoM detection in the same month = highest priority.  
Ranked by combined z-score (average of cross-sectional max_z and MoM max_z).

In [ ]:
def compute_combined_scores(cs_df, mom_df, dim):
    """Merge cross-sectional and MoM results, find entities flagged by both."""
    if len(mom_df) == 0:
        return pd.DataFrame()
    
    # Merge on entity + month
    merge_cols = [dim, 'hce_admit_month']
    cs_subset = cs_df[merge_cols + ['max_abs_z', 'cs_flagged', 'cs_top_feature']].copy()
    mom_subset = mom_df[merge_cols + ['max_abs_mom_z', 'mom_flagged', 'mom_top_feature']].copy()
    
    combined = cs_subset.merge(mom_subset, on = merge_cols, how = 'inner')
    
    # Both flagged
    combined['both_flagged'] = combined['cs_flagged'] & combined['mom_flagged']
    combined['combined_z'] = (combined['max_abs_z'] + combined['max_abs_mom_z']) / 2
    
    return combined


combined_results = {}
for dim in DIMENSIONS:
    combined_results[dim] = compute_combined_scores(cs_results[dim], mom_results[dim], dim)

# Summary
for dim in DIMENSIONS:
    cdf = combined_results[dim]
    if len(cdf) > 0:
        both = cdf[cdf['both_flagged']]
        print(f'{dim}: {len(both)} entity-months flagged by BOTH methods '
              f'({both[dim].nunique()} distinct entities)')
    else:
        print(f'{dim}: No combined results (insufficient MoM history)')

In [ ]:
# Top combined-flagged entities per dimension
for dim in DIMENSIONS:
    cdf = combined_results[dim]
    if len(cdf) == 0:
        continue
    both = cdf[cdf['both_flagged']].copy()
    if len(both) == 0:
        print(f'{dim}: No entities flagged by both methods')
        continue
    
    # Rank by combined_z, show top entities
    top_combined = (
        both
        .groupby(dim)
        .agg(n_months_both = ('hce_admit_month', 'count'),
             avg_combined_z = ('combined_z', 'mean'),
             max_combined_z = ('combined_z', 'max'),
             cs_top = ('cs_top_feature', lambda x: x.mode().iloc[0] if len(x) > 0 else ''),
             mom_top = ('mom_top_feature', lambda x: x.mode().iloc[0] if len(x) > 0 else ''))
        .sort_values('avg_combined_z', ascending = False)
        .head(15)
    )
    print(f'\n=== {dim} — Top Combined-Flagged Entities ===')
    display(top_combined)

## 7. Feature-Level Deep Dive

For each feature in priority order, show the top outlier entities across all dimensions.  
This helps identify which specific behavior each entity is anomalous on.

**Feature priority**: persistent_adr_rate → initial_adr_rate → member_appeal_overturn_rate → p2p_overturn_rate → member_appeal_rate → p2p_rate → auth_per_k

In [ ]:
for feat in PRIORITY_FEATURES:
    direction = FLAG_DIRECTION[feat]
    print(f'\n{"="*80}')
    print(f'FEATURE: {feat}  (suspicious direction: {direction})')
    print(f'{"="*80}')
    
    z_col = f'{feat}_z'
    
    for dim in DIMENSIONS:
        cs_df = cs_results[dim]
        if z_col not in cs_df.columns:
            continue
        
        # Filter to directionally-flagged entities only
        if direction == 'low':
            flagged_mask = cs_df[z_col] <= -MAD_Z_THRESHOLD
        else:
            flagged_mask = cs_df[z_col] >= MAD_Z_THRESHOLD
        
        # Sort: 'low' features ascending (most negative first), 'high' descending
        sort_ascending = (direction == 'low')
        
        top_for_feat = (
            cs_df[flagged_mask]
            .sort_values(z_col, ascending = sort_ascending)
            .groupby(dim)
            .agg(
                n_months_outlier = ('hce_admit_month', 'count'),
                avg_z = (z_col, 'mean'),
                avg_value = (feat, 'mean'),
                avg_cases = ('case_count', 'mean'),
            )
            .sort_values('avg_z', ascending = sort_ascending)
            .head(10)
        )
        if len(top_for_feat) > 0:
            print(f'\n  {dim} (top outliers for {feat}):')
            display(top_for_feat)

## 8. Distribution Profiling — Monthly Features

Histograms, correlations, skewness, and sparsity on the monthly profiles.

In [ ]:
# Histograms — priority features across dimensions (using most recent complete month)
latest_month = all_months[-2]  # second-to-last for completeness
print(f'Distribution profiling for month: {latest_month}')

for dim in DIMENSIONS:
    prof = monthly_profiles[dim]
    month_slice = prof[prof['hce_admit_month'] == latest_month]
    
    n_feats = len(PRIORITY_FEATURES)
    ncols = 4
    nrows = (n_feats + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize = (16, 4 * nrows))
    fig.suptitle(f'{dim} | month={latest_month} (n={len(month_slice)})', fontsize = 13)
    
    for ax, feat in zip(axes.flatten(), PRIORITY_FEATURES):
        vals = month_slice[feat].dropna()
        ax.hist(vals, bins = 30, edgecolor = 'white', alpha = 0.8, color = cmc.batlow(0.55))
        ax.set_title(feat, fontsize = 10)
        if len(vals) > 0:
            ax.axvline(vals.median(), color = cmc.vik(0.95), linestyle = '--', label = f'med={vals.median():.3f}')
            ax.legend(fontsize = 8)
    
    # Hide unused axes
    for ax in axes.flatten()[n_feats:]:
        ax.set_visible(False)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation heatmap — prov_tin (highest N)
tin_latest = monthly_profiles['prov_tin'][monthly_profiles['prov_tin']['hce_admit_month'] == latest_month]
df_corr = tin_latest[PRIORITY_FEATURES].corr()

fig, ax = plt.subplots(figsize = (10, 8))
sns.heatmap(df_corr, annot = True, fmt = '.2f', cmap = cmc.vik, center = 0,
            square = True, linewidths = 0.5, ax = ax, annot_kws = {'size': 9})
ax.set_title(f'Feature Correlation — Provider TIN, month={latest_month}', fontsize = 13)
plt.tight_layout()
plt.show()

In [ ]:
# Skewness across all dimensions and months
skew_summary = pd.DataFrame({
    dim: monthly_profiles[dim][PRIORITY_FEATURES].skew().round(2)
    for dim in DIMENSIONS
})
skew_summary

In [ ]:
# Sparsity — % of entity-months with zero values
sparsity_summary = pd.DataFrame({
    dim: ((monthly_profiles[dim][PRIORITY_FEATURES] == 0).sum() / len(monthly_profiles[dim]) * 100).round(1)
    for dim in DIMENSIONS
})
sparsity_summary

## 9. Summary & Next Steps

### Observations Framework

After reviewing the analyses above, note:

1. **Cross-sectional flags**: Which entities are consistently outliers across multiple months? These are structurally different from peers.  
2. **MoM flags**: Which entities show sudden spikes relative to their own history? These may indicate recent behavior changes.  
3. **Combined flags**: Entities appearing in both — structurally anomalous AND recently spiking — are the highest-priority suspects.  
4. **Feature concentration**: Are anomalies concentrated on specific features (e.g., persistent_adr_rate) or spread across multiple?  
5. **Dimension patterns**: Do market-level anomalies explain hospital_group or TIN anomalies (hierarchical effect)?  

### Decision Points

- **Threshold tuning**: |z| > 2.5 may be too strict or too lenient — examine the flag rates above.  
- **Rolling window**: 4-month trailing may need adjustment depending on seasonality.  
- **Minimum cases**: 15 per entity-month — check if this excludes too many entities at the TIN level.  
- **Hierarchical drill**: If a market is flagged, decompose into its hospital groups and TINs to find the driver.  
- **Next step**: Move from EDA to a scoring pipeline that produces monthly watchlists.